In [5]:
import pandas as pd
import re
import base64

# Load data
df = pd.read_csv("Data.csv", dtype=str)  # Ensure all data is treated as string
df = df.fillna('')  # Fill any NaN values with empty strings

# Combine all text from the dataframe into a single string
text = ' '.join(df.astype(str).apply(' '.join, axis=1))

# Regex patterns for different IP forms
ipv4_pattern = r'\b(?:(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)\.){3}(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)\b'
ipv6_pattern = r'\b(?:[A-Fa-f0-9]{1,4}:){7}[A-Fa-f0-9]{1,4}\b'
ipv4_url_pattern = r'https?:\/\/(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)(?:\.(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)){3}\b'
ipv6_url_pattern = r'https?:\/\/\[(?:[A-Fa-f0-9]{1,4}:){7}[A-Fa-f0-9]{1,4}\]'
obfuscated_ip_pattern = r'\b(?:\d{1,3})[\.\[\]\(\)](?:\d{1,3})[\.\[\]\(\)](?:\d{1,3})[\.\[\]\(\)](?:\d{1,3})\b'
loose_ip_pattern = r'\b(?:\d{1,3}\.){3}\d{1,3}\b'  # Matches sequences like 192.168.1.1 even if some parts are invalid

# Step 1: Detect and replace obfuscated IP formats
text = re.sub(r'\[dot\]|\(dot\)|\{dot\}', '.', text)

# Step 2: Check for split IPs across columns (reconstruct split IPs)
split_ip_pattern = r'(\d+)[.,]?\s*(\d+)[.,]?\s*(\d+)[.,]?\s*(\d+)'  # Matches IPs that are split across columns
text = re.sub(split_ip_pattern, r'\1.\2.\3.\4', text)

# Step 3: Check for Base64 encoded IPs
def decode_base64_and_extract_ip(encoded_str):
    try:
        # Try decoding the base64 string
        decoded_str = base64.b64decode(encoded_str).decode('utf-8', 'ignore')
        # If it decodes successfully, check for IP-like patterns
        return re.findall(ipv4_pattern, decoded_str) + re.findall(ipv6_pattern, decoded_str)
    except Exception:
        return []

# Extract Base64-encoded substrings and check if they decode into IPs
base64_ips = re.findall(r'[A-Za-z0-9+/=]{8,}', text)
decoded_ips = []
for base64_str in base64_ips:
    decoded_ips += decode_base64_and_extract_ip(base64_str)

# Step 4: Find all matches using the regex patterns
all_ips = set()
all_ips.update(re.findall(ipv4_pattern, text))
all_ips.update(re.findall(ipv6_pattern, text))
all_ips.update(re.findall(ipv4_url_pattern, text))
all_ips.update(re.findall(ipv6_url_pattern, text))
all_ips.update(re.findall(obfuscated_ip_pattern, text))
all_ips.update(re.findall(loose_ip_pattern, text))
all_ips.update(decoded_ips)  # Add Base64 decoded IPs

# Output results
print("Total unique IP-like matches found:", len(all_ips))
print("IPs found:", list(all_ips)[:-1])


Total unique IP-like matches found: 80
IPs found: ['1.9.7.0', '2.7.4.2', '63.2.7.1', '1.9.8.3', '26.4.201.9', '2.0.1.0', '2.0.0.3', '1.9.8.5', '2.0.2.0', '1.9.0.6', '2.4.3.8', '6.3.9.7', '1.9.9.3', '2.0.2.2', '1.0.6.1', '8.20.2.1', '1.9.3.7', '1.9.4.7', '19.20.2.3', '287.25.4.4', '7.8.3.1', '1.3.1.2', '1.9.7.6', '1.9.7.4', '2.2.3.3', '3.9.0.5', '1.0.8.8', '2.0.1.7', '150.0.0.0', '3.9.0.8', '3.8.1.9', '2.0.0.8', '1.9.9.7', '5.1.6.0', '2.0.0.7', '7.0.0.0', '1.0.0.7', '100.0.0.0', '403.6.8.5', '8.1.1.1', '929.2.9.2', '0.0.0.1', '1.9.9.9', '2.0.1.1', '3.8.0.0', '1.9.2.6', '1.9.7.9', '2.9.0.0', '1.9.6.1', '7.6.2.5', '2.0.1.6', '2.0.1.5', '1.9.0.5', '2.0.0.5', '28.8.8.1', '501.5.0.2', '2.0.1.3', '1.9.7.7', '1.9.8.1', '1.9.0.1', '1.9.8.8', '2.0.1.8', '2.0.2.4', '6.5.8.7', '28.20.2.2', '7.1.0.0', '2.8.0.5', '9.4.1.8', '8.4.7.2', '2.0.2.1', '2.7.5.7', '37.0.0.0', '0.0.0.0', '2.0.0.0', '1.6.3.4', '1.800.66.3', '1.8.8.5', '1.9.6.2', '5.5.5.5']
